# Forest Place Recognition -- VPR Benchmark Demo

This notebook demonstrates **forest-place-recognition**, a benchmark framework for comparing
Visual Place Recognition (VPR) methods on forest and seasonal datasets.

The framework provides:
- Pluggable feature-extraction backends
- Cosine-similarity matching with top-K retrieval
- Evaluation metrics: Recall@K, precision-recall curves, average precision

**Supported backends:**

| Backend | Descriptor dim | GPU required | Description |
|---------|---------------|-------------|-------------|
| `histogram` | 80 | No | HSV color histogram baseline |
| `resnet_gem` | 2048 | Optional | ResNet-50 + GeM pooling |
| `eigenplaces` | 512 | Yes | EigenPlaces (pretrained) |
| `cosplace` | 512 | Yes | CosPlace (pretrained) |

## 1. Setup

In [ ]:
%matplotlib inline

import time
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from forest_place_recognition.features import FeatureExtractor
from forest_place_recognition.matcher import cosine_similarity_matrix, top_k_retrieval
from forest_place_recognition.evaluation import (
    compute_recall_at_k,
    compute_precision_recall,
    recall_at_multiple_k,
    average_precision,
)
from forest_place_recognition.backends import available_backends, BACKEND_NAMES

print("All registered backends:", BACKEND_NAMES)
print("Available backends (deps installed):", available_backends())

## 2. Create synthetic forest images

We generate 10 synthetic images with random green/brown patterns to simulate forest scenes.
This lets the demo run without the FinnWoodlands dataset.

In [ ]:
def make_forest_image(rng: np.random.Generator, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Generate a synthetic forest-like image with green/brown blobs."""
    h, w = size
    img = np.zeros((h, w, 3), dtype=np.uint8)

    # Sky-ish top gradient
    sky_h = rng.integers(h // 6, h // 3)
    for y in range(sky_h):
        ratio = y / sky_h
        img[y, :] = [int(135 + 50 * ratio), int(180 + 30 * ratio), int(220 + 20 * ratio)]

    # Fill the rest with random green/brown patches
    for _ in range(rng.integers(15, 40)):
        cx, cy = rng.integers(0, w), rng.integers(sky_h, h)
        rx, ry = rng.integers(10, 60), rng.integers(10, 60)
        # Random green or brown
        if rng.random() < 0.6:
            color = [rng.integers(10, 80), rng.integers(80, 200), rng.integers(10, 60)]
        else:
            color = [rng.integers(80, 160), rng.integers(50, 120), rng.integers(10, 50)]

        y_lo, y_hi = max(0, cy - ry), min(h, cy + ry)
        x_lo, x_hi = max(0, cx - rx), min(w, cx + rx)
        # Blend
        alpha = rng.uniform(0.3, 0.9)
        patch = img[y_lo:y_hi, x_lo:x_hi].astype(float)
        patch = patch * (1 - alpha) + np.array(color, dtype=float) * alpha
        img[y_lo:y_hi, x_lo:x_hi] = patch.clip(0, 255).astype(np.uint8)

    return img


# Generate and save images
tmp_dir = Path(tempfile.mkdtemp(prefix="forest_vpr_demo_"))
rng = np.random.default_rng(42)
n_images = 10
image_paths: list[Path] = []

for i in range(n_images):
    arr = make_forest_image(rng)
    p = tmp_dir / f"forest_{i:03d}.png"
    Image.fromarray(arr).save(p)
    image_paths.append(p)

print(f"Saved {n_images} images to {tmp_dir}")

# Display grid
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, p in zip(axes.flat, image_paths):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=9)
    ax.axis("off")
fig.suptitle("Synthetic Forest Images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Feature extraction comparison

We compare the `histogram` (fast, CPU-only) and `resnet_gem` (deep learning) backends.

In [ ]:
results = {}

for backend_name in ["histogram", "resnet_gem"]:
    print(f"\n--- {backend_name} ---")
    extractor = FeatureExtractor(backend=backend_name)

    t0 = time.perf_counter()
    descriptors = extractor.extract_batch(image_paths)
    elapsed = time.perf_counter() - t0

    sparsity = float(np.mean(np.abs(descriptors) < 1e-6))

    results[backend_name] = {
        "descriptors": descriptors,
        "dim": descriptors.shape[1],
        "time": elapsed,
        "mean": float(np.mean(descriptors)),
        "std": float(np.std(descriptors)),
        "sparsity": sparsity,
    }

    print(f"  Descriptor dim : {descriptors.shape[1]}")
    print(f"  Extraction time: {elapsed:.3f} s ({elapsed / n_images * 1000:.1f} ms/image)")
    print(f"  Mean / Std     : {np.mean(descriptors):.4f} / {np.std(descriptors):.4f}")
    print(f"  Sparsity       : {sparsity:.2%}")

## 4. Matching demo

Compute the cosine similarity matrix and retrieve the top-3 matches for each query.

In [ ]:
# Use histogram descriptors for the matching demo
descs = results["histogram"]["descriptors"]

# Split into query (first 5) and reference (last 5)
query_desc = descs[:5]
ref_desc = descs[5:]
query_paths = image_paths[:5]
ref_paths = image_paths[5:]

sim_matrix = cosine_similarity_matrix(query_desc, ref_desc)

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xlabel("Reference index")
ax.set_ylabel("Query index")
ax.set_title("Cosine Similarity Matrix")
ax.set_xticks(range(len(ref_paths)))
ax.set_yticks(range(len(query_paths)))
ax.set_xticklabels([p.stem for p in ref_paths], fontsize=8, rotation=45)
ax.set_yticklabels([p.stem for p in query_paths], fontsize=8)
for i in range(sim_matrix.shape[0]):
    for j in range(sim_matrix.shape[1]):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Top-3 retrieval
top_k = 3
indices, scores = top_k_retrieval(sim_matrix, k=top_k)

fig, axes = plt.subplots(len(query_paths), top_k + 1, figsize=(12, 3 * len(query_paths)))
for i, q_path in enumerate(query_paths):
    axes[i, 0].imshow(Image.open(q_path))
    axes[i, 0].set_title(f"Query: {q_path.stem}", fontsize=9)
    axes[i, 0].axis("off")
    for j in range(top_k):
        ref_idx = indices[i, j]
        axes[i, j + 1].imshow(Image.open(ref_paths[ref_idx]))
        axes[i, j + 1].set_title(f"#{j+1} {ref_paths[ref_idx].stem}\nscore={scores[i, j]:.3f}", fontsize=8)
        axes[i, j + 1].axis("off")

fig.suptitle("Top-3 Retrieved Matches", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Evaluation metrics

We create a synthetic ground-truth matrix and compute Recall@K and precision-recall curves.

In [ ]:
# Use all 10 images as both query and reference (self-retrieval)
all_desc = results["histogram"]["descriptors"]
sim_all = cosine_similarity_matrix(all_desc, all_desc)

# Build synthetic ground truth: identity (each image matches itself)
# plus random "nearby" matches to simulate realistic scenarios
gt = np.eye(n_images, dtype=bool)
# Add adjacent images as valid matches (simulating temporal proximity)
for i in range(n_images - 1):
    gt[i, i + 1] = True
    gt[i + 1, i] = True

# Retrieve top-10
indices_all, scores_all = top_k_retrieval(sim_all, k=10)

# Recall@K
recalls = recall_at_multiple_k(indices_all, gt, ks=[1, 5, 10])
print("Recall@K results:")
for k, v in recalls.items():
    print(f"  Recall@{k}: {v:.2%}")

# Average precision
ap = average_precision(
    scores=scores_all[:, 0],
    top1_indices=indices_all[:, 0],
    ground_truth=gt,
)
print(f"\nAverage Precision: {ap:.4f}")

In [ ]:
# Precision-recall curve
precision, recall, thresholds = compute_precision_recall(
    scores=scores_all[:, 0],
    top1_indices=indices_all[:, 0],
    ground_truth=gt,
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, "b-", linewidth=2)
ax.fill_between(recall, precision, alpha=0.15)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (AP = {ap:.3f})")
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Backend comparison table

Run all available backends, compare descriptor dimensions and extraction speed.

In [ ]:
import pandas as pd

backends_to_test = available_backends()
print(f"Testing backends: {backends_to_test}\n")

rows = []
for name in backends_to_test:
    ext = FeatureExtractor(backend=name)

    t0 = time.perf_counter()
    desc = ext.extract_batch(image_paths)
    elapsed = time.perf_counter() - t0

    rows.append({
        "Backend": name,
        "Descriptor dim": desc.shape[1],
        "Total time (s)": round(elapsed, 3),
        "ms / image": round(elapsed / n_images * 1000, 1),
        "Mean": round(float(np.mean(desc)), 4),
        "Std": round(float(np.std(desc)), 4),
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
# Bar chart: extraction speed per backend
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(df["Backend"], df["ms / image"], color=["#2ecc71", "#3498db", "#e67e22", "#e74c3c"][:len(df)])
ax.set_ylabel("ms / image")
ax.set_title("Feature Extraction Speed by Backend")
for bar, val in zip(bars, df["ms / image"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## 7. Summary

### Adding custom backends

To add a new VPR backend:

1. Create a class that implements `extract(image_path) -> np.ndarray` and the `descriptor_dim` property.  
   See `forest_place_recognition/backends/base.py` for the abstract interface.
2. Register it in `forest_place_recognition/backends/__init__.py` by adding an entry to `_BACKENDS`.

### FinnForest dataset

For real-world evaluation, download the **FinnForest** / **FinnWoodlands** dataset:  
https://github.com/oravus/FinnForest

Use `forest_place_recognition.loader` to load image paths and GPS positions,
then run the same evaluation pipeline shown above with `compute_recall_at_k_from_distances()`
for GPS-based ground truth.

In [ ]:
# Clean up temp directory
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print("Temporary images cleaned up.")